In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error
from matplotlib import pyplot as plt

# Configuração para exibir todas as colunas que vamos criar
pd.set_option('display.max_columns', None)

print("Ambiente configurado com sucesso! 🚀")

Ambiente configurado com sucesso! 🚀


In [5]:
#Importando os dados trabalhados do EDA
df = pd.read_csv('dados.csv', delimiter=',', header='infer')
df['data'] = pd.to_datetime(df['data'])
df = df[df['data'] >= '2000-01-01'].copy()
df = df.sort_values('data').reset_index(drop=True)

print(f"Formato do dataframe: {df.shape[0]} meses registrados.")
df.head()

Formato do dataframe: 316 meses registrados.


,data,ipca,selic,cambio,igpm
0,2000-01-01,0.62,18.94,1.8037,1.24
1,2000-02-01,0.13,18.87,1.7753,0.35
2,2000-03-01,0.22,18.85,1.7420,0.15
3,2000-04-01,0.42,18.62,1.7682,0.23
4,2000-05-01,0.01,18.51,1.8279,0.31


XGBoost, ele não entende o conceito de tempo nativamente.

* primeiro passo é extrair informações numéricas da própria data.
    * mes
    * ano

In [6]:
#criando features de calendário
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month

df.head()

,data,ipca,selic,cambio,igpm,ano,mes
0,2000-01-01,0.62,18.94,1.8037,1.24,2000,1
1,2000-02-01,0.13,18.87,1.7753,0.35,2000,2
2,2000-03-01,0.22,18.85,1.7420,0.15,2000,3
3,2000-04-01,0.42,18.62,1.7682,0.23,2000,4
4,2000-05-01,0.01,18.51,1.8279,0.31,2000,5


criação de lags ou Defasagens
 * se quisermos que ele preveja o IPCA do mês que vem, precisamos dar a ele pistas do passado recente na mesma linha

 * lag1 -> ipca do mês anterior 
 * lag2 -> ipca de dois meses atrás (base para justificativa do lag1, pois ele captura a tendência de curto prazo)
 * lag12 -> ipca de 1 ano atrás (captura a sazonalidade anual de forma direta).

In [9]:
#Lags para o IPCA 
df['ipca_lag1'] = df['ipca'].shift(1)
df['ipca_lag2'] = df['ipca'].shift(2)
df['ipca_lag12'] = df['ipca'].shift(12)

# Criando os Lags para as variáveis exógenas (Selic, Câmbio e igpm)
df['selic_lag_1'] = df['selic'].shift(1)
df['selic_lag_2'] = df['selic'].shift(2)
df['selic_lag_3'] = df['selic'].shift(3)

df['cambio_lag_1'] = df['cambio'].shift(1)
df['cambio_lag_2'] = df['cambio'].shift(2)
df['cambio_lag_3'] = df['cambio'].shift(3)

df['igpm_lag_1'] = df['igpm'].shift(1)
df['igpm_lag_2'] = df['igpm'].shift(2)
df['igpm_lag_3'] = df['igpm'].shift(3)

df.head(5)

,data,ipca,selic,cambio,igpm,ano,mes,ipca_lag1,ipca_lag2,ipca_lag12,selic_lag_1,selic_lag_2,selic_lag_3,cambio_lag_1,cambio_lag_2,cambio_lag_3,igpm_lag_1,igpm_lag_2,igpm_lag_3
0,2000-01-01,0.62,18.94,1.8037,1.24,2000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,0.13,18.87,1.7753,0.35,2000,2,0.62,NaN,NaN,18.94,NaN,NaN,1.8037,NaN,NaN,1.24,NaN,NaN
2,2000-03-01,0.22,18.85,1.7420,0.15,2000,3,0.13,0.62,NaN,18.87,18.94,NaN,1.7753,1.8037,NaN,0.35,1.24,NaN
3,2000-04-01,0.42,18.62,1.7682,0.23,2000,4,0.22,0.13,NaN,18.85,18.87,18.94,1.7420,1.7753,1.8037,0.15,0.35,1.24
4,2000-05-01,0.01,18.51,1.8279,0.31,2000,5,0.42,0.22,NaN,18.62,18.85,18.87,1.7682,1.7420,1.7753,0.23,0.15,0.35
